<img src="../../img/backdrop-wh.png" alt="Drawing" style="width: 300px;"/>

# AI Coding Audit: Delegation and Hermeneutic Debt

* * *


<div class="alert alert-success">

### Learning Objectives

* Probe the "jagged frontier" of LLM coding ability with an open-ended prompt.
* Identify the interpretive decisions an LLM makes for you when you delegate an analysis.
* Write a bounded prompt that modifies a known method, and explain the result in your own words.
* Verify AI-generated code, and understand verification as repaying hermeneutic debt.
* Decide deliberately how much of your final-project workflow to delegate.

</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

### Sections
1. [Orientation: The Tool You Are Probably Already Using](#orient)
2. [Privacy and Tool Choice](#privacy)
3. [Delegation and Hermeneutic Debt](#debt)
4. [Load the Data](#load)
5. [Choose Your Method and Run a Baseline](#method)
6. [Round 1: Open Delegation](#round1)
7. [Round 2: Bounded Collaboration](#round2)
8. [Compare the Rounds](#compare)
9. [Key Points and Audit Checklist](#final)


<a id='orient'></a>

# 1. Orientation

Many of you have probably already used an LLM for coding in this course, whether to debug an exercise or to make sense of an error message at 11pm. This is just what working with code looks like now. This week, that existing practice becomes the object of study.

If you have been doing everything yourself, the weeks you spent on preprocessing, TF-IDF, topic modeling, and embeddings now allow you to *supervise* the delegation. You can only probe what a model can do if you can evaluate what comes back.

Today you will run the same task two ways: once by handing the model as much freedom as possible, and once by constraining it tightly. You will then study what each approach affords. 

### How this notebook relates to the discussion post

This notebook links to the **Week 5 AI Coding Audit** discussion post on bCourses. Each Student Action below points to the discussion section it links to, so make sure to take notes.

<a id='privacy'></a>

# 2. Privacy and Tool Choice

⚠️ **Warning:** Do not paste private, sensitive, identifying, or unpublished data into an AI system. 

| Tool | Notes |
|------|-------|
| **UC-licensed Gemini** | Recommended default. Campus-supported, no paid account needed. [gemini.google.com](https://gemini.google.com) with your Berkeley login. |
| **ChatGPT** | Acceptable. Free tier is sufficient. |
| **Claude** | Acceptable. Free tier is sufficient. |
| **Coding agents** (Claude Code, Cursor, Codex, Antigravity) | Allowed. Agents write *and run* code on their own, so keep a record of what you asked for and what the agent did along the way; you will need both for the audit. |
| **GitHub Copilot** (autocomplete) | Allowed but not recommended: the prompt-response trail is harder to document for the audit. |

### Student Action: Tool Choice

*Notes for discussion section 3: Which tool are you using, and are there any privacy considerations to keep in mind with it?*


<a id='debt'></a>

# 3. Delegation and Hermeneutic Debt

Current models are good at coding, good enough to write most of the code in this course unaided. Getting working code is rarely the obstacle anymore. We will try to answer **what happens to interpretation when you delegate the coding?**

### The jagged frontier

LLM capability is uneven in ways that are hard to predict. A model may flawlessly implement a topic model but interpret the topic labels in a very unintuitive way. 

This has been called the [**jagged frontier**](https://www.hbs.edu/faculty/Pages/item.aspx?num=64700). The only way to find these edges is to poke at a model. One way to do this is with an underspecified prompt: asking for "the most interesting analysis of this data" shows you what the model can really do, including things you would not have thought to ask for.

### Hermeneutic debt

However, every decision you leave unspecified is a decision the model makes for you; which method, which thresholds, which categories, and what counts as "interesting." 

We can call the accumulation of these decisions **hermeneutic debt**. These are interpretive choices made on your behalf that you have not yet examined. Like financial debt, it can get you somewhere fast, but you can only repay it by tracing, explaining, and verifying what the model did. Left unpaid, it compounds inside the claims you make.

Coding agents sit further along that spectrum than chatbots. A chatbot writes code that you copy/paste into your notebook and run yourself. An agent (Claude Code, Cursor, Codex, Antigravity) also runs the code, reads the errors, and fixes them on its own. Each pass through that loop involves decisions you never see. 

If you work with an agent this week, the exercise stays the same. The debt simply accrues faster.

### What you hand over when you ask

| When you ask... | You hand over... |
|---|---|
| "Explain this code / this error / this parameter" | Almost nothing: every answer can be checked against your own code. |
| "Adapt this code to compare two groups" | Implementation choices. The model fills every gap you leave with its own defaults. |
| "Do the most interesting analysis of my data" | Method, categories, thresholds, and what the model considers *interesting*. |
| "What do these results mean? (without including scripts/data)" | Debt you cannot repay by verification: the model hasn't read your data and cannot check its own claims. |

🔔 **Question:** Think of one prompt you have actually sent an LLM in the past five weeks. Where does it sit in this table? What did you hand over?


<a id='load'></a>

# 4. Load the Data

Ground everything in an actual dataset before you involve an AI. All packages below are part of the course Conda environment; if you see an ImportError on DataHub, uncomment the pip line and restart your kernel.


In [1]:
# Uncomment only if you see an ImportError on DataHub:
# %pip install pandas scikit-learn gensim

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
import gdown

file_id = "1rAnEFVYAMu_DVcZNK3pl7WI4meeMvVVg"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "../../data/aita_pp.csv", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1rAnEFVYAMu_DVcZNK3pl7WI4meeMvVVg
To: /Users/tomvannuenen/Library/CloudStorage/Dropbox/GitHub/DEV/DIGHUM160/data/aita_pp.csv
100%|██████████| 56.6M/56.6M [00:01<00:00, 39.2MB/s]


'../../data/aita_pp.csv'

In [3]:
df = pd.read_csv('../../data/aita_pp.csv')

print("Columns:", list(df.columns))
print(f"Number of rows: {len(df)}")
print("\nText column preview:")
print(df['selftext'].iloc[0][:300])

Columns: ['idint', 'idstr', 'created', 'self', 'nsfw', 'author', 'title', 'url', 'selftext', 'score', 'subreddit', 'distinguish', 'textlen', 'num_comments', 'flair_text', 'flair_css_class', 'augmented_at', 'augmented_count', 'pp_text']
Number of rows: 16309

Text column preview:
My girlfriend recently went to the beach with a few of her friends.  She has this tiny bikini bottom that is basically a thong that I HATE when she wears in public.  Well she wore it.  Not only did she wear it, she posed in the bathroom mirror of her hotel room to take a side profile picture so you 


💡 **Tip:** If you want to work with your own dataset from a different subreddit, update the file path accordingly.

### Student Action: Dataset Description

*Notes for discussion section 1: What does one row in your dataset represent? What is one **interpretive question** this dataset might help you explore?*


<a id='method'></a>

# 5. Choose Your Method and Run a Baseline

Pick **one** method below, choosing by the interpretive question you noted in Section 4. 

Each track answers a different kind of question. Pick a track and run only that cell. The starter code is taken from the Week 2, 3, and 4 lesson notebooks, so you have seen it before. This **baseline** is your anchor. In Round 2 you will ask the model to modify it, and in both rounds it gives you something known to compare against.

⚠️ **Warning:** Do not modify the starter code yet, and only run the track you chose.

| Track | Plain-English description | Choose it if your question is about... |
|---|---|---|
| **A: Distinctive language** | Scores every word by how characteristic it is of the posts (TF-IDF), and shows the top terms. | Recurring vocabulary, rhetorical patterns, what words define this community. |
| **B: Topics** | Groups words that tend to appear together into "topics" (LDA), and shows the top words per topic. | Recurring themes or situations across many posts. |
| **C: Embeddings** | Trains a word-embedding model (Word2Vec) that places words with similar contexts near each other, and shows a word's nearest neighbors. | What a concept means in this community: which words travel together, how a term is used *here*. |


In [4]:
# ── TRACK A: DISTINCTIVE LANGUAGE (from the Week 2 lesson) ───────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_df=0.85,
                                   max_features=1000,
                                   decode_error='ignore',
                                   stop_words='english',
                                   smooth_idf=True,
                                   use_idf=True)

# Fit and transform the texts
tfidf = tfidf_vectorizer.fit_transform(df['pp_text'])

# Place TF-IDF values in a DataFrame
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame.sparse.from_spmatrix(tfidf, columns=feature_names)

# Highest TF-IDF values across documents
tfidf_df.sum().sort_values(ascending=False)

said             901.539467
told             839.344899
like             746.327548
wife             635.564836
got              585.513201
                    ...    
jack               41.04934
tomorrow          41.048533
lawyer            41.035679
understanding     40.966328
anna              38.416699
Length: 1000, dtype: Sparse[float64, 0]

In [5]:
# ── TRACK B: TOPICS (from the Week 3 lesson) ─────────────────────────────────
from gensim import corpora
from gensim.models.ldamodel import LdaModel

lemmas_split = [lemma.split() for lemma in df['pp_text']]

# Create Dictionary
dictionary = corpora.Dictionary(lemmas_split)

# Filter extremes and assign new ids
dictionary.filter_extremes(no_below=10, no_above=0.4)
dictionary.compactify()

# Create Document-Term Matrix of our whole corpus
corpus = [dictionary.doc2bow(text) for text in lemmas_split]

# Train the model (takes 1-2 minutes)
lda_model = LdaModel(corpus=corpus,   # stream of document vectors
            id2word=dictionary,       # mapping from word IDs to words
            num_topics=10,            # amount of topics
            random_state=100,         # seed to generate random state
            passes=2,                 # amount of iterations/epochs
            per_word_topics=False)

# Show the top words for each topic
for idx, topic in lda_model.print_topics(num_topics=10, num_words=10):
    print(f"Topic {idx}: {topic}\n")

Topic 0: 0.042*"daughter" + 0.024*"school" + 0.010*"kids" + 0.008*"parents" + 0.007*"class" + 0.007*"sister" + 0.006*"teacher" + 0.005*"year" + 0.004*"things" + 0.004*"husband"

Topic 1: 0.019*"money" + 0.017*"kids" + 0.016*"house" + 0.013*"pay" + 0.012*"parents" + 0.010*"job" + 0.009*"home" + 0.009*"work" + 0.008*"sister" + 0.008*"live"

Topic 2: 0.013*"food" + 0.010*"home" + 0.009*"eat" + 0.008*"dinner" + 0.007*"husband" + 0.007*"night" + 0.006*"day" + 0.006*"room" + 0.006*"sleep" + 0.005*"house"

Topic 3: 0.027*"work" + 0.011*"day" + 0.010*"job" + 0.007*"home" + 0.007*"office" + 0.006*"working" + 0.006*"company" + 0.005*"week" + 0.005*"people" + 0.005*"went"

Topic 4: 0.065*"son" + 0.022*"husband" + 0.015*"wife" + 0.013*"mil" + 0.012*"sil" + 0.011*"ring" + 0.008*"kids" + 0.008*"year_old" + 0.007*"baby" + 0.007*"cake"

Topic 5: 0.040*"wife" + 0.021*"money" + 0.018*"car" + 0.014*"$" + 0.011*"christmas" + 0.009*"pay" + 0.007*"family" + 0.007*"gift" + 0.006*"year" + 0.006*"buy"

Topic 6

In [6]:
# ── TRACK C: WORD EMBEDDINGS (from the Week 4 lesson) ────────────────────────
from gensim.models import Word2Vec
import multiprocessing

# Split posts into lists of words for Word2Vec
post_list = [post.split() for post in df['pp_text']]

cores = min(4, multiprocessing.cpu_count())

n_features = 300     # Word vector dimensionality (how many features each word will be given)
min_word_count = 10  # Minimum word count to be taken into account
window = 5           # Context window size
downsampling = 1e-2  # Downsample setting for frequent words
seed = 1             # Seed for the random number generator (to create reproducible results)
sg = 1               # Skip-gram = 1, CBOW = 0

# Train the model (takes a few minutes)
model = Word2Vec(
    sentences=post_list,
    workers=cores,
    vector_size=n_features,
    min_count=min_word_count,
    window=window,
    sample=downsampling,
    seed=seed,
    sg=sg)

# Look up the terms most similar to a word (try your own!)
for word, similarity in model.wv.most_similar(positive=['wedding'], topn=10):
    print(f"{word}: {round(similarity, 3)}")

ceremony: 0.696
reception: 0.695
engagement_party: 0.676
bride_groom: 0.674
bridesmaid: 0.671
bridal_party: 0.665
moh: 0.661
walking_aisle: 0.659
destination_wedding: 0.657
venue: 0.656


### Warm-Up: Ask the Model to Explain the Baseline

The starter code was written for you (like it has been in previous weeks). Let's start with the lowest-debt request, and ask a model what the code means. 

Paste your track's code into your LLM and ask it to explain what each step does. Check the explanation against the output you just got, and against the relevant lesson notebook (Week 2 for Track A, Week 3 for Track B, Week 4 for Track C). 

### Student Action: Document the Baseline

*Notes for discussion section 2: After the warm-up, what did the baseline produce, in your own words? Note one pattern or limitation you see; you will compare against this in both rounds.*


<a id='round1'></a>

# 6. Round 1: Open Delegation

Now, let's probe the frontier. Give the model your dataset's shape and your research interest, then hand over as much as possible. For example:

```
I have a pandas dataframe called df with [N] Reddit posts from r/[subreddit].
Each row is one post; the text is in the column 'selftext' [list other columns].
I am interested in [your interpretive question, in one sentence].

Propose and write the most interesting analysis of this data you can think of.
Use only pandas, numpy, scikit-learn, gensim, and matplotlib. Return complete,
runnable code.
```

Notice what this prompt leaves unspecified: the method, the parameters, the output, and what "interesting" means. This is deliberate. We are maximizing both discovery and debt.

⚠️ **Warning:** Read the code before running it. If it imports a library you have not seen, look it up first. If it references a column that does not exist in `df`, that is a hallucination; note it for your audit.

💡 **Tip:** If you are using a coding agent rather than a chat window, point it at your data file, give it the same brief, and let it work. Then paste the final code it produced into the cell below, and note what it did on its own along the way. This can include errors it hit and fixed, packages it chose, or steps it added without asking. 

### Student Action: Round 1 Prompt

*Notes for discussion section 3: Paste the actual prompt you sent, verbatim.*


In [ ]:
# ── ROUND 1: PASTE THE MODEL'S CODE HERE ─────────────────────────────────────
# Read it first. Then run it. If it errors, record the error before fixing.

# [paste code here]

### Student Action: Debt Audit

*Notes for discussion section 5: The model just made a series of interpretive decisions you never asked for. Find them.*

**Which method did it choose, and why do you think it chose that one?**

**List every decision it made that you did not specify** (parameters, thresholds, filtering, categories, what it treated as "interesting"):

**Where was the frontier jagged?** What did it nail? What did it get wrong, invent, or oversimplify?

**Did it run on the first try?** If not, paste the error and what you changed:

💭 **Reflection:** The output may be genuinely interesting. Can you defend it? For each decision on your list, could you explain to a peer why it was the right choice for your question, or is it debt you are still carrying?


<a id='round2'></a>

# 7. Round 2: Bounded Collaboration

Now the other end of the spectrum. Ask the model to make **one specific modification** to your baseline from Section 5. You decide what changes; the model implements it.

A bounded prompt specifies: your research question, the dataframe and column names, the current code (paste it), the *one* change you want, library constraints, and a request to explain each step. For example:

| Weak (open) | Bounded |
|---|---|
| "Make this better." | "Modify the TF-IDF code below to compare top terms between posts flaired NTA and posts flaired YTA. Use pandas and scikit-learn only. Comment each step." |
| "Find the themes." | "The LDA code below produces 5 topics. Change it to 8 topics and print the 3 most representative posts for each. Do not change the preprocessing." |
| "What does this word mean here?" | "The Word2Vec code below shows the 10 nearest neighbors of 'wedding'. Change it to compare the neighbors of 'mother' and 'father', and explain what the similarity scores measure." |

### Student Action: Round 2 Prompt

*Notes for discussion section 3: Paste the actual prompt you sent, verbatim, and one sentence on what you expected back.*


In [ ]:
# ── ROUND 2: PASTE THE MODEL'S CODE HERE ─────────────────────────────────────
# Read it first. Then run it.

# [paste code here]

### Student Action: Round 2 Log

*Notes for discussion sections 4 and 5: Work through the questions below.*

**Did it run on the first try?** If not, what was the error and the fix?

**Did the model stay inside your constraints?** Note anything it introduced unasked (libraries, variables, extra steps).

**Explain what the code does at the level of decisions**: what goes in, what the code does to it, what comes out, and which choices are embedded along the way (parameters, thresholds, filters). You may ask the model to explain its own code, since explanation requests carry little debt. Check any explanation it gives you against the actual code and output before you adopt it.


### Repay the Debt: Verify

Code that runs without errors is not necessarily correct. Pick **two** checks and run them below. This is the repayment step:

- Manually inspect rows the output claims something about.
- Check that a term/topic/result actually appears in the underlying posts.
- Compare the modified output against your Section 5 baseline: what changed, and does the change make sense?
- Re-run with one alternate parameter value and see how stable the result is.
- Confirm no hallucinated columns, labels, or numbers.

You can ask the LLM to help *write* a verification check. Asking it to judge whether its own output is correct is another matter. That debt has no repayment mechanism: a real check runs against the data rather than the model's confidence.


In [ ]:
# ── VERIFICATION ─────────────────────────────────────────────────────────────
# Two checks against the actual data. Examples:
#
# Check a term actually appears:
#   matching = df[df['selftext'].str.contains('some_term', case=False, na=False)]
#   print(len(matching)); print(matching['selftext'].iloc[0][:300])
#
# Compare with baseline: re-run your Section 5 cell and diff the outputs by eye.

# [your verification code here]

### Student Action: Verification Notes

*Notes for discussion section 5: What did you check? What became more trustworthy? What do you still not fully trust?*


<a id='compare'></a>

# 8. Compare the Rounds

You now have two versions of AI-assisted analysis: one where you handed over the interpretive core, and one where you kept it. Both are legitimate ways to work; they carry different debt.

### Student Action: Comparison

*Notes for discussion sections 3 and 5: These notes are the core of your post.*

**Which round produced the more interesting result?**

**Which result can you actually explain and defend, choice by choice?**

**What did Round 1 surface that you would never have asked for, and what would you need to examine before you could build a claim on it?**

**What new questions did the two rounds open that you did not start with?**

💭 **Reflection:** For your final project, where on the delegation spectrum will you work, and how will you repay what you borrow? There is no house style here: an open probe followed by careful auditing is a legitimate workflow, and so is tight constraint from the start. The only unacceptable option is carrying the debt silently into your claims.


<a id='final'></a>

<div class="alert alert-success">

## ❗ Key Points

* Models can write most of this course's code. The question is what happens to interpretation when they do.
* The frontier is jagged, and you find its edges by probing. Open prompts are probes.
* Every unspecified decision is a decision the model makes for you. That is hermeneutic debt, and it is repaid by tracing, explaining, and verifying what the model did.
* Understanding the methods (Weeks 1–4) is what makes delegation safe: it lets you supervise the model instead of trusting it.
* AI-generated code is allowed. Unexplained code is not.

</div>

### Audit Checklist → Week 5 Discussion Post

Everything the discussion post needs is now in your notes. Before closing this notebook, check that you can locate:

| Discussion section | Where it is in this notebook |
|---|---|
| 1. Research question and dataset | Section 4 dataset description |
| 2. Starting point | Section 5 baseline + your notes |
| 3. Delegation choices and prompts | Both prompts (Sections 6 and 7) |
| 4. Generated or revised code | Your Round 1 or Round 2 excerpt + your own-words explanation |
| 5. Verification and debt | Debt audit (Section 6), Round 2 log, verification notes |
